In [19]:
import os
import json
import csv

# --- Configuration ---
# Path to your data
base_path = '/home/literature/hindwi-scraper/output'
poets_metadata_file = os.path.join(base_path, 'poets.json')
all_poets_dir = os.path.join(base_path, 'poets')

# Path for the final output dataset file
output_csv_path = 'kaal_prediction_dataset.csv'

print("✅ Libraries imported and paths are set.")
print(f"Output dataset will be saved to: {output_csv_path}")

✅ Libraries imported and paths are set.
Output dataset will be saved to: kaal_prediction_dataset.csv


In [20]:
poet_kaal_map = {}

try:
    with open(poets_metadata_file, 'r', encoding='utf-8') as f:
        poets_data = json.load(f)

    # Create the slug -> kaal map, skipping poets without a valid kaal
    for poet in poets_data:
        # Ensure the 'kaal' key exists and is not an empty string
        if poet.get('kaal') and poet.get('kaal').strip():
            poet_kaal_map[poet['poet_slug']] = poet['kaal']
            
    print(f"✅ Kaal map created for {len(poet_kaal_map)} poets with valid labels.")

except FileNotFoundError:
    print(f"❌ Error: The main poets file was not found at {poets_metadata_file}")

✅ Kaal map created for 1133 poets with valid labels.


In [21]:
poems_in_dataset = 0

# Open the new CSV file in write mode
with open(output_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    # Create a CSV writer object
    csv_writer = csv.writer(csvfile)
    
    # Write the header row for the columns 'text' and 'label'
    csv_writer.writerow(['text', 'label'])
    
    # Loop through each poet's directory using the map we created
    for poet_slug, kaal in poet_kaal_map.items():
        poet_dir = os.path.join(all_poets_dir, poet_slug)
        poems_dir = os.path.join(poet_dir, 'kavita')
        
        # Check if the poet's 'kavita' directory exists
        if os.path.isdir(poems_dir):
            for filename in os.listdir(poems_dir):
                # Process only the Devanagari text files
                if filename.endswith('-devnagri.txt'):
                    filepath = os.path.join(poems_dir, filename)
                    
                    try:
                        # Read the full content of the poem file
                        with open(filepath, 'r', encoding='utf-8') as f:
                            poem_text = f.read()
                        
                        # Write the poem text and its kaal (label) to the CSV
                        if poem_text.strip(): # Ensure the poem file is not empty
                            csv_writer.writerow([poem_text, kaal])
                            poems_in_dataset += 1
                        
                    except Exception as e:
                        print(f"Error reading file {filepath}: {e}")

print("\n---")
print(f"🎉 Success! Dataset created at: {output_csv_path}")
print(f"Total poems added to the dataset: {poems_in_dataset}")


---
🎉 Success! Dataset created at: kaal_prediction_dataset.csv
Total poems added to the dataset: 15279


In [22]:
import json

# --- Configuration ---
poets_metadata_file = '/home/literature/hindwi-scraper/output/poets.json'
poets_without_kaal = 0
total_poets = 0

try:
    with open(poets_metadata_file, 'r', encoding='utf-8') as f:
        poets_data = json.load(f)
        total_poets = len(poets_data)

    # Iterate through each poet's data to check for a valid 'kaal'
    for poet in poets_data:
        # Check if 'kaal' key is missing, is None, or is an empty string
        if not poet.get('kaal') or not poet.get('kaal').strip():
            poets_without_kaal += 1
            
    print("--- Data Audit Complete ---")
    print(f"Total poets in original file: {total_poets}")
    print(f"Poets with no 'kaal' provided: {poets_without_kaal}")

except FileNotFoundError:
    print(f"❌ Error: The file was not found at {poets_metadata_file}")

--- Data Audit Complete ---
Total poets in original file: 2978
Poets with no 'kaal' provided: 1845


In [23]:
import pandas as pd

# Define the path to your dataset
csv_file_path = 'kaal_prediction_dataset.csv'

try:
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(csv_file_path)
    
    # Use the value_counts() method on the 'label' column to get the counts
    kaal_counts = df['label'].value_counts()
    
    print("--- Poem Count per Kaal ---")
    print(kaal_counts)
    
except FileNotFoundError:
    print(f"❌ Error: The dataset file was not found at {csv_file_path}")

--- Poem Count per Kaal ---
label
आधुनिक काल    15238
भक्तिकाल         40
रीतिकाल           1
Name: count, dtype: int64


In [18]:
import pandas as pd

# Define the path to your original poets metadata file
json_file_path = '/home/literature/hindwi-scraper/output/poets.json'

try:
    # Read the JSON file into a pandas DataFrame
    df = pd.read_json(json_file_path)
    
    # Remove any poets that do not have a 'kaal' assigned
    df_filtered = df.dropna(subset=['kaal'])
    
    # Use value_counts() on the 'kaal' column to get the poet counts
    poet_counts_per_kaal = df_filtered['kaal'].value_counts()
    
    print("--- Poet Count per Kaal ---")
    print(poet_counts_per_kaal)
    
except FileNotFoundError:
    print(f"❌ Error: The file was not found at {json_file_path}")

--- Poet Count per Kaal ---
kaal
आधुनिक काल    835
रीतिकाल       161
भक्तिकाल      107
आदिकाल         30
Name: count, dtype: int64


In [24]:
import pandas as pd
import numpy as np

# Define the path to your original poets metadata file
json_file_path = '/home/literature/hindwi-scraper/output/poets.json'

try:
    # Read the JSON file into a pandas DataFrame
    df = pd.read_json(json_file_path)

    # --- Data Cleaning and Preparation ---

    # A function to safely extract years from the 'birth_death' list
    def extract_year(date_list, position):
        try:
            # Check if the list is long enough and the item is a digit string
            if len(date_list) > position and isinstance(date_list[position], str) and date_list[position].isdigit():
                return int(date_list[position])
        except (TypeError, ValueError):
            return np.nan # Return Not a Number for invalid data
        return np.nan

    # Create 'birth_year' and 'death_year' columns
    df['birth_year'] = df['birth_death'].apply(lambda x: extract_year(x, 0))
    df['death_year'] = df['birth_death'].apply(lambda x: extract_year(x, 1))

    # Drop poets who do not have a kaal or a valid birth year
    df_cleaned = df.dropna(subset=['kaal', 'birth_year'])
    df_cleaned['birth_year'] = df_cleaned['birth_year'].astype(int)

    # --- Calculation ---

    print("--- Approximate Time Periods Based on Your Data ---")

    # Group by 'kaal' and find the min birth year and max of birth/death years
    kaal_groups = df_cleaned.groupby('kaal')

    for name, group in kaal_groups:
        start_year = int(group['birth_year'].min())
        
        # The end year is the latest known year, either birth or death
        end_year = int(group[['birth_year', 'death_year']].max().max())
        
        # Handle modern era poets who might still be alive
        if name == 'आधुनिक काल':
            period_str = f"{start_year} – Present"
        else:
            period_str = f"{start_year} – {end_year}"
            
        print(f"{name:<12}: {period_str}")

except FileNotFoundError:
    print(f"❌ Error: The file was not found at {json_file_path}")

--- Approximate Time Periods Based on Your Data ---
आदिकाल      : 780 – 1460
आधुनिक काल  : 1833 – Present
भक्तिकाल    : 1270 – 1958
रीतिकाल     : 1538 – 1921


/tmp/ipykernel_488087/2765750499.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['birth_year'] = df_cleaned['birth_year'].astype(int)


In [26]:
import os
import json
from collections import defaultdict

# --- Configuration ---
# Define the base directory where your 'output' folder is located.
base_path = '/home/literature/hindwi-scraper/output'
poets_metadata_file = os.path.join(base_path, 'poets.json')
all_poets_dir = os.path.join(base_path, 'poets')

# Define the academic time periods for each Kaal
KAAL_PERIODS = {
    'आदिकाल': (0, 1375),
    'भक्तिकाल': (1375, 1700),
    'रीतिकाल': (1700, 1900),
    'आधुनिक काल': (1900, 9999) # From 1900 onwards
}

def get_kaal_from_year(year):
    """Assigns a Kaal based on the poet's birth year."""
    if not year or not isinstance(year, int):
        return 'अज्ञात (वर्ष नहीं मिला)'
    for kaal, (start, end) in KAAL_PERIODS.items():
        if start <= year < end:
            return kaal
    return 'अज्ञात (वर्ष सीमा से बाहर)'

def analyze_poets_and_poems():
    """
    Counts poets and their Devanagari poems, categorizing them by Kaal
    based on their birth year.
    """
    try:
        with open(poets_metadata_file, 'r', encoding='utf-8') as f:
            poets_data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Error: The file was not found at {poets_metadata_file}")
        return

    poet_counts = defaultdict(int)
    poem_counts = defaultdict(int)

    for poet in poets_data:
        birth_year = None
        if poet.get('birth_death') and len(poet['birth_death']) > 0:
            try:
                year_str = str(poet['birth_death'][0]).split(' ')[0]
                if year_str.isdigit():
                    birth_year = int(year_str)
            except (ValueError, TypeError):
                birth_year = None
        
        kaal = get_kaal_from_year(birth_year)
        poet_counts[kaal] += 1

        poet_slug = poet.get('poet_slug')
        if not poet_slug:
            continue

        poems_dir = os.path.join(all_poets_dir, poet_slug, 'kavita')
        
        if os.path.isdir(poems_dir):
            devnagri_poem_count = 0
            for filename in os.listdir(poems_dir):
                if filename.endswith('-devnagri.txt'):
                    devnagri_poem_count += 1
            poem_counts[kaal] += devnagri_poem_count

    # --- Print the final report ---
    print("--- काव्यों और कवियों की काल-अनुसार गणना ---")
    print("-" * 75)
    print(f"{'काल':<15} | {'समय-अवधि':<18} | {'कवियों की संख्या':<20} | {'कविताओं की संख्या'}")
    print("-" * 75)

    sorted_kaals = sorted(poet_counts.keys())

    for kaal in sorted_kaals:
        num_poets = poet_counts[kaal]
        num_poems = poem_counts[kaal]
        
        # Get and format the time period for the current kaal
        period = KAAL_PERIODS.get(kaal)
        if period:
            start, end = period
            if kaal == 'आधुनिक काल':
                period_str = f"{start} – Present"
            else:
                period_str = f"{start} – {end}"
        else:
            period_str = "N/A"

        print(f"{kaal:<15} | {period_str:<18} | {num_poets:<20} | {num_poems}")
        
    print("-" * 75)

# --- Run the analysis ---
analyze_poets_and_poems()

--- काव्यों और कवियों की काल-अनुसार गणना ---
---------------------------------------------------------------------------
काल             | समय-अवधि           | कवियों की संख्या     | कविताओं की संख्या
---------------------------------------------------------------------------
अज्ञात (वर्ष नहीं मिला) | N/A                | 647                  | 2137
अज्ञात (वर्ष सीमा से बाहर) | N/A                | 1                    | 1
आदिकाल          | 0 – 1375           | 34                   | 1
आधुनिक काल      | 1900 – Present     | 1725                 | 18427
भक्तिकाल        | 1375 – 1700        | 131                  | 13
रीतिकाल         | 1700 – 1900        | 440                  | 960
---------------------------------------------------------------------------
